In [ ]:
# Rutas portables: ejecutar desde notebooks/ o desde la raíz clonada.
from pathlib import Path
import sys, os
_repo = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs/pipeline.json').exists())
sys.path.insert(0, str(_repo / 'src'))
from pipeline_runtime import prepare_runtime
_runtime = prepare_runtime(_repo)
os.chdir(_runtime)
sys.path.insert(0, str(_runtime))

import json
_central = json.loads((_repo / 'configs/pipeline.json').read_text(encoding='utf-8'))


# 3.0.0 — Análisis bivariado y selección de cortes WoE

Continuación independiente de los artefactos finales del PSI. No ejecuta el notebook 2.0, filtros previos ni PSI. Los originales v4 se conservan para XGBoost; los cortes y WoE se aprenden **solo con TRAIN**. No se entrenan modelos predictivos ni se realiza selección multivariada.

El módulo reutilizable se llama **`3.0.0.bivariado.py`**, según la convención solicitada. El prefijo numérico requiere la carga explícita de BIV.1. No copie sus funciones al notebook.

**Primera preparación (ya realizada para esta entrega):**
```powershell
.\.venv\Scripts\python.exe 3.0.0.bivariado.py export-v4
```
Esta utilidad verifica la fuente y recupera la selección PSI persistida, sin repetir cálculos. Después el notebook necesita ?nicamente `BIVARIADO/entradas/` y el módulo. Ejecute BIV.1?BIV.14 para revisar y **BIV.15 cuando desee exportar**.

Pruebas y automatización opcional desde la raíz:
```powershell
.\.venv\Scripts\python.exe -m unittest test_bivariado -v
.\.venv\Scripts\python.exe 3.0.1.ejecutar_bivariado.py
```
El ejecutor captura tablas y `plt.show()` mediante Python; no inicia un kernel Jupyter. Las pruebas no ejecutan ni modifican notebooks.


**Política B.3:** se conservan búsqueda y fusiones B.2, con copias previas en `evidencias/B3_base.json`.
La aprobación requiere >=2 bins ordinarios, IV total >=0.02, tendencia numérica estricta, 5% por bin ordinario, suficientes buenos/malos y diferenciación >=0.5 puntos porcentuales.
No se aplica preferencia por >=3 bins ni umbral adicional de AUC a dos bins.
Se mantienen los controles mensuales de suficiencia/estabilidad existentes. Los estados finales son APROBADA, RECHAZADA y NO_EVALUABLE; esta ?ltima señala datos o errores que impiden una conclusión.
Los parámetros históricos B.2 solo se utilizan en las pruebas de compatibilidad explícitas, no en este notebook.

Los nuevos paneles aprobados TRAIN/TEST/OOT emplean el mismo orden de variables/bins, páginas y escalas. TEST/OOT solo aportan observaciones diagnósticas; no participan en FIT.


## BIV.1 — Importación de librerías y configuración

**Objetivo:** Cargar el módulo de funciones y declarar parámetros auditables.

**Procedencia:** `3.0.0.bivariado.py`.

**Resultado esperado:** Módulo cargado y tabla de parámetros, sin lectura de fuentes anteriores.

Los umbrales iniciales son revisables; no garantizan estabilidad fuera de muestra. No hay filtros automáticos por IV>0.30 ni por no monotonicidad.

In [ ]:
from pathlib import Path
import importlib.util
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# Archivo: 3.0.0.bivariado.py. El nombre numérico impide un import convencional.
ruta_modulo = Path("3.0.0.bivariado.py")
especificacion = importlib.util.spec_from_file_location("bivariado_3_0", ruta_modulo)
biv = importlib.util.module_from_spec(especificacion)
especificacion.loader.exec_module(biv)

# Función: parameters(). Valida y entrega una configuración independiente.
PARAMETROS_BIV = biv.parameters(
    POLICY_VERSION="B.3", Q_VALUES=list(range(2, 11)), ALPHA=0.5,
    MIN_IV_TOTAL=0.02, MIN_BIN_PCT=0.05, MIN_BIN_N=200,
    MIN_GOODS=20, MIN_BADS=10, MIN_RATE_GAP=0.005,
    MIN_FIT_OBS=400, MIN_EVAL_BIN_N=50,
    MIN_MONTH_BIN_N=50, MIN_MONTH_BIN_EVENTS=3,
    MIN_TEMPORAL_COVERAGE=0.70, TEMPORAL_Z_ALERT=3.0,
    MAX_TEMPORAL_ALERT_FRACTION=0.20,
    IV_RETAIN_FRACTION=0.90, MERGE_ADJACENT=True,
    UNSEEN_POLICY="training_prior_woe_0",
)
# B.3 no aplica MIN_PREFERRED_BINS ni los filtros AUC/IV/gap especiales de dos bins.
display(pd.DataFrame(biv.active_parameters(PARAMETROS_BIV).items(), columns=["Parametro", "Valor"]))
PARAMETROS_BIV = biv.parameters(**_central["bivariate"])


## BIV.2 — Carga de resultados del PSI y validación de muestras

**Objetivo:** Recuperar matrices, targets, identificadores y periodos persistidos y versionados.

**Procedencia:** `3.0.0.bivariado.py`.

**Resultado esperado:** Diagnóstico por segmento, tipos y estados PSI; ?ndices y hashes validados.



In [ ]:
# Archivo: 3.0.0.bivariado.py | Funciones: load_v4(), sample_diagnostics().
# No lee la base original ni ejecuta notebook 2.0; verifica manifest y hashes.
muestras_biv = biv.load_v4()
X_train_v4 = muestras_biv["X"]["TRAIN"]
X_test_v4 = muestras_biv["X"]["TEST"]
X_oot_v4 = muestras_biv["X"]["OOT"]
y_train = muestras_biv["y"]["TRAIN"]
y_test = muestras_biv["y"]["TEST"]
y_oot = muestras_biv["y"]["OOT"]
metadata_train = muestras_biv["metadata"]["TRAIN"]
metadata_test = muestras_biv["metadata"]["TEST"]
metadata_oot = muestras_biv["metadata"]["OOT"]
print("Versión de entrada:", muestras_biv["manifest"]["version"])
print(f"Variables reales: {X_train_v4.shape[1]}; referencia esperada: 29 (sin forzar).")
display(biv.sample_diagnostics(muestras_biv))
display(X_train_v4.dtypes.rename("Tipo").to_frame())
display(muestras_biv["psi_summary"][["Variable", "Estado_PSI"]])

## BIV.3 — Análisis exploratorio de variables candidatas

**Objetivo:** Describir variabilidad, nulos, ceros e infinitos de TRAIN, sin filtrar.

**Procedencia:** `3.0.0.bivariado.py`.

**Resultado esperado:** Una fila por variable candidata.



In [ ]:
# Archivo: 3.0.0.bivariado.py | Función: explore_train().
df_exploratorio_bivariado = biv.explore_train(X_train_v4)
display(df_exploratorio_bivariado)

## BIV.4 — Generación de candidatos q=2 a 10

**Objetivo:** Aprender límites candidatos exclusivamente en X_train_v4.

**Procedencia:** `3.0.0.bivariado.py`.

**Resultado esperado:** Todos los q solicitados, límites reales y bins definidos.



In [ ]:
# Archivo: 3.0.0.bivariado.py | Funciones: generate_candidates(), candidate_inventory().
# Los discretos de baja cardinalidad usan categorías reales; q repetidos se conservan.
candidatos_iniciales_biv = biv.generate_candidates(X_train_v4, PARAMETROS_BIV)
display(biv.candidate_inventory(candidatos_iniciales_biv))
print("Configuraciones iniciales:", len(candidatos_iniciales_biv))

## BIV.5 — Cálculo de WoE e IV por agrupación

**Objetivo:** Contar buenos/malos y estimar WoE, IV, AUC/Gini/KS y tasas mensuales de cada candidato.

**Procedencia:** `3.0.0.bivariado.py`.

**Resultado esperado:** Reportes completos y registro de configuraciones originales/fusionadas.

WoE = ln(P(bin|bueno)/P(bin|malo)); IV = ?(P_bueno?P_malo)?WoE. ALPHA suaviza conteos de bins observados y normaliza distribuciones. AUC/Gini/KS usan ?WoE: son diagnósticos aparentes de desarrollo y pueden ser optimistas; no se ajusta regresión logística.
B.2 recalcula tasas, WoE e IV desde conteos después de cada fusión. Solo combina vecinos y conserva su trazabilidad; no promedia estimaciones.


In [ ]:
# Archivo: 3.0.0.bivariado.py | Función: evaluate_candidates().
# Internamente: assign_bins(), bin_statistics(), monthly_rates(), search_adjacent_partitions().
# Solo recibe TRAIN; ninguna muestra TEST/OOT participa en el aprendizaje.
(df_candidatos_binning, df_bins_detalle, df_estabilidad_candidatos,
 registro_candidatos_biv) = biv.evaluate_candidates(
    X_train_v4, y_train, metadata_train, candidatos_iniciales_biv, PARAMETROS_BIV
)
display(df_bins_detalle)
print(f"Evaluadas {len(df_candidatos_binning)} configuraciones; "
      f"{len(df_bins_detalle)} bins observados en TRAIN.")
# B.2: la búsqueda examina todas las particiones contiguas y ambos sentidos.
# Se conserva la frontera de calidad/IV por dirección y número de bins,
# además de todos los candidatos q originales, incluso los no factibles.
display(df_candidatos_binning[[
    "Variable", "Candidate_ID", "N_particiones_exploradas", "N_particiones_factibles",
    "N_fusiones", "Min_pct_ordinario", "Cumple_5pct", "Estado_candidato", "Motivos"
]])


## BIV.6 — Evaluación de monotonicidad y calidad

**Objetivo:** Revisar volumen, eventos, tendencia, cobertura y alertas por candidato.

**Procedencia:** `3.0.0.bivariado.py`.

**Resultado esperado:** Todos los motivos de aceptación/revisión sin excluir variables.

La tendencia excluye bins especiales y no asigna orden artificial a categorías nominales. Los nulos escasos generan una alerta de seguimiento, quedan exentos de mínimos ordinarios y no se mezclan con cero ni con intervalos.
Las tasas planas son NO_EVALUABLE; la tendencia aproximada se distingue de la estricta. Una variable nominal carece de orden numérico y no se fuerza una dirección.


In [ ]:
# Resultados de evaluate_candidates() / monotonicity() en 3.0.0.bivariado.py.
display(df_candidatos_binning[[
    "Variable", "Candidate_ID", "q_solicitado", "N_bins_efectivos",
    "N_bins_insuficientes", "N_bins_ordinarios_insuficientes", "Alerta_missing_escaso", "N_bins_puros", "Min_buenos", "Min_malos",
    "Monotonicidad", "Monotonicidad_estricta", "Spearman_tasas_orden", "IV_ordinario", "IV_missing", "Cumple_5pct", "Cobertura_temporal",
    "Fraccion_alertas_tasas", "Estado_candidato", "Motivos", "Observaciones"
]])

## BIV.7 — Ranking de variables por IV

**Objetivo:** Comparar el IV máximo de desarrollo antes de elegir una configuración preferida.

**Procedencia:** `3.0.0.bivariado.py`.

**Resultado esperado:** Ranking exploratorio; no implica escoger el máximo IV.



In [ ]:
# Reporte derivado de evaluate_candidates(), procedente de 3.0.0.bivariado.py.
df_ranking_iv_candidatos = (
    df_candidatos_binning.sort_values("IV_total", ascending=False)
    .drop_duplicates("Variable")
    [["Variable", "Candidate_ID", "IV_total", "AUC_train_desarrollo",
      "Gini_train_desarrollo", "KS_train_desarrollo", "Estado_candidato"]]
)
display(df_ranking_iv_candidatos.reset_index(drop=True))

## BIV.8 — Selección de cortes por variable

**Objetivo:** Elegir una configuración considerando calidad, estabilidad TRAIN y simplicidad.

**Procedencia:** `3.0.0.bivariado.py`.

**Resultado esperado:** Resumen preferido y artefacto WoE en memoria, con estados y motivos.

RECHAZADA / NO_EVALUABLE conserva una configuración provisional. No elimina la variable ni afirma que sea apta para producción. No se usa TEST/OOT para decidir.
Incluye comparación B.2/B.3 para todas las variables y una tabla separada para las variables previamente en revisión. Se identifica pérdida IV >20% como alerta, no como meta a optimizar.


In [ ]:
# Archivo: 3.0.0.bivariado.py | Función: select_configurations().
# Calidad primero; dentro del margen de IV, prefiere menos bins.
df_resumen_bivariado, binning_woe_train = biv.select_configurations(
    df_candidatos_binning, df_bins_detalle, registro_candidatos_biv,
    metadata_train, PARAMETROS_BIV
)
variables_aprobadas_bivariado = binning_woe_train["approved_variables"].copy()
variables_rechazadas_bivariado = binning_woe_train["rejected_variables"].copy()
variables_no_evaluables_bivariado = binning_woe_train["not_evaluable_variables"].copy()
display(df_resumen_bivariado)
display(df_resumen_bivariado.groupby(["Estado_bivariado", "Monotonicidad"]).size().rename("Variables").to_frame())
print("Estados finales congelados exclusivamente con TRAIN.")
# Archivo: 3.0.0.bivariado.py | Función: compare_binning().
# La comparación histórica NO participa en el aprendizaje ni en la selección.
# Añade al artefacto versionado B.3 los cortes anteriores y los definitivos.
df_comparacion_binning = biv.compare_binning(
    df_resumen_bivariado, df_bins_detalle, binning_woe_train
)
display(df_comparacion_binning)
df_siete_revision_anterior = df_comparacion_binning.loc[
    df_comparacion_binning.Anteriormente_en_revision
].copy()
print("Resultado de las variables que requerían revisión en B.2:")
display(df_siete_revision_anterior)


## BIV.9 — Tabla detallada de cortes seleccionados

**Objetivo:** Inspeccionar cada bin, sus límites exactos y estadísticas.

**Procedencia:** `3.0.0.bivariado.py`.

**Resultado esperado:** Detalle de cortes, grupos, conteos, tasas, WoE e IV preferidos.



In [ ]:
# Selección del reporte bin_statistics() generado en 3.0.0.bivariado.py.
df_cortes_seleccionados = df_bins_detalle.loc[
    df_bins_detalle.Candidate_ID.isin(df_resumen_bivariado.Candidate_ID)
].copy()
display(df_cortes_seleccionados)
# El JSON conserva también el mapa de categorías reales y los límites exactos,
# aunque la etiqueta del gráfico redondee solo para facilitar la lectura.
display(df_cortes_seleccionados[[
    "Variable", "Bin", "Limite_inferior", "Limite_superior", "N", "Buenos", "Malos",
    "Dist_clientes", "Tasa_default", "WoE", "IV_bin", "Es_missing", "Exento_min_pct"
]])


## BIV.10 — Gráfico individual: volumen y tasa de malos

**Objetivo:** Elegir una variable y revisar la distribución y tasa de default por bin.

**Procedencia:** `3.0.0.bivariado.py`.

**Resultado esperado:** Un gráfico seleccionado, sin escribir archivos.


El bin de ausencia aparece al final, sombreado y con punto independiente: la línea roja no lo conecta con los intervalos numéricos. El título indica bins efectivos y fusiones.


In [ ]:
VARIABLE_BIV = df_resumen_bivariado.Variable.iloc[0]  # Cambie el nombre para explorar.
# Archivo: 3.0.0.bivariado.py | Función: plot_variable().
figura, _, _ = biv.plot_variable(VARIABLE_BIV, df_cortes_seleccionados, df_resumen_bivariado)
figura.tight_layout()
plt.show()
plt.close(figura)

## BIV.11 — Dashboard TRAIN - todas las variables

**Objetivo:** Revisar varias variables preferidas en cada página.

**Procedencia:** `3.0.0.bivariado.py`.

**Resultado esperado:** Páginas de hasta 12 variables, cuatro columnas y leyenda general.



In [ ]:
ESTADO_DASHBOARD_BIV = "TODAS"  # TODAS / APROBADA / RECHAZADA / NO_EVALUABLE
# Archivo: 3.0.0.bivariado.py | Función: plot_dashboard().
figuras_dashboard_biv = biv.plot_dashboard(
    df_cortes_seleccionados, df_resumen_bivariado,
    estado=ESTADO_DASHBOARD_BIV, columnas=4, por_pagina=12
)
if figuras_dashboard_biv:
    plt.show()
    for figura in figuras_dashboard_biv:
        plt.close(figura)
else:
    print("No hay variables con ese estado.")

## BIV.12 — Estabilidad mensual de la tasa de malos en TRAIN

**Objetivo:** Examinar volumen, eventos, tasas, intervalos de confianza y alertas mensuales.

**Procedencia:** `3.0.0.bivariado.py`.

**Resultado esperado:** Reporte de bins seleccionados por mes y gráfico de una variable.

Las alertas Z comparan con los otros meses de TRAIN y son diagnósticas, sin corrección por comparaciones múltiples. Los bins/meses escasos quedan identificados; no se dibujan como tasas cero.

In [ ]:
# Reporte de monthly_rates() en 3.0.0.bivariado.py; ya calculado para cada candidato.
df_estabilidad_tasas = df_estabilidad_candidatos.loc[
    df_estabilidad_candidatos.Candidate_ID.isin(df_resumen_bivariado.Candidate_ID)
].copy()
display(df_estabilidad_tasas)
# Archivo: 3.0.0.bivariado.py | Función: plot_monthly_rates().
figura, _ = biv.plot_monthly_rates(VARIABLE_BIV, df_estabilidad_tasas, df_resumen_bivariado)
plt.show()
plt.close(figura)

### BIV.12A - Estadísticas observadas con cortes TRAIN congelados

Procedencia: `3.0.0.bivariado.py`. Objetivo: obtener conteos, tasas reales y advertencias por volumen en las tres muestras con exactamente los mismos bins.

Se utilizan dataframes en memoria, sin leer PNG anteriores ni ejecutar FIT.

In [ ]:
# Archivo: 3.0.0.bivariado.py | Función: evaluate_frozen_samples().
# Solo agrupa registros y calcula tasas observadas; no calcula nuevos WoE.
(df_estadisticas_bivariado_muestras, df_comparacion_bivariado_muestras,
 df_resumen_bivariado_muestras) = biv.evaluate_frozen_samples(muestras_biv, binning_woe_train)
display(df_resumen_bivariado_muestras)

### BIV.12B - Dashboard TRAIN - solo aprobadas

Procedencia: `3.0.0.bivariado.py`. Las barras representan % de clientes y la línea roja tasas observadas. Un bin vacío conserva su lugar y tasa NaN; los nulos se muestran separados. Bajo volumen genera una advertencia, sin cambiar cortes.

Se utilizan dataframes en memoria, sin leer PNG anteriores ni ejecutar FIT.

In [ ]:
# Archivo: 3.0.0.bivariado.py | Función: plot_sample_dashboard().
# La lista y el orden proceden del JSON aprobado en TRAIN; escalas comunes por variable.
figuras_train_aprobadas = biv.plot_sample_dashboard(
    df_estadisticas_bivariado_muestras, df_resumen_bivariado,
    binning_woe_train, segmento="TRAIN", columnas=4, por_pagina=12
)
if figuras_train_aprobadas:
    plt.show()
    for figura in figuras_train_aprobadas:
        plt.close(figura)
else:
    print("No hay variables APROBADA en TRAIN.")

### BIV.12C - Dashboard TEST - solo aprobadas

Procedencia: `3.0.0.bivariado.py`. Las barras representan % de clientes y la línea roja tasas observadas. Un bin vacío conserva su lugar y tasa NaN; los nulos se muestran separados. Bajo volumen genera una advertencia, sin cambiar cortes.

Se utilizan dataframes en memoria, sin leer PNG anteriores ni ejecutar FIT.

In [ ]:
# Archivo: 3.0.0.bivariado.py | Función: plot_sample_dashboard().
# La lista y el orden proceden del JSON aprobado en TRAIN; escalas comunes por variable.
figuras_test_aprobadas = biv.plot_sample_dashboard(
    df_estadisticas_bivariado_muestras, df_resumen_bivariado,
    binning_woe_train, segmento="TEST", columnas=4, por_pagina=12
)
if figuras_test_aprobadas:
    plt.show()
    for figura in figuras_test_aprobadas:
        plt.close(figura)
else:
    print("No hay variables APROBADA en TRAIN.")

### BIV.12D - Dashboard OOT - solo aprobadas

Procedencia: `3.0.0.bivariado.py`. Las barras representan % de clientes y la línea roja tasas observadas. Un bin vacío conserva su lugar y tasa NaN; los nulos se muestran separados. Bajo volumen genera una advertencia, sin cambiar cortes.

Se utilizan dataframes en memoria, sin leer PNG anteriores ni ejecutar FIT.

In [ ]:
# Archivo: 3.0.0.bivariado.py | Función: plot_sample_dashboard().
# La lista y el orden proceden del JSON aprobado en TRAIN; escalas comunes por variable.
figuras_oot_aprobadas = biv.plot_sample_dashboard(
    df_estadisticas_bivariado_muestras, df_resumen_bivariado,
    binning_woe_train, segmento="OOT", columnas=4, por_pagina=12
)
if figuras_oot_aprobadas:
    plt.show()
    for figura in figuras_oot_aprobadas:
        plt.close(figura)
else:
    print("No hay variables APROBADA en TRAIN.")

### BIV.12E - Comparación estadística TRAIN / TEST / OOT

Procedencia: `3.0.0.bivariado.py`. Objetivo: comparar los bins ya definidos sin reajuste.

Se utilizan dataframes en memoria, sin leer PNG anteriores ni ejecutar FIT.

In [ ]:
# Resultado de evaluate_frozen_samples(), procedente de 3.0.0.bivariado.py.
# Tasas en proporción; diferencias en puntos porcentuales. No se utilizan para seleccionar.
display(df_comparacion_bivariado_muestras)
display(df_resumen_bivariado_muestras)

### BIV.12F - Lista final aprobada / rechazada / no evaluable

Procedencia: `3.0.0.bivariado.py`. Las variables rechazadas y no evaluables quedan fuera de las matrices WoE aprobadas, conservando candidatos y métricas de auditoría.

Se utilizan dataframes en memoria, sin leer PNG anteriores ni ejecutar FIT.

In [ ]:
# Listas de select_configurations() / 3.0.0.bivariado.py, calculadas antes de evaluar TEST/OOT.
display(pd.DataFrame([
    {"Variable": v, "Estado_bivariado": estado}
    for estado, variables in [
        ("APROBADA", variables_aprobadas_bivariado),
        ("RECHAZADA", variables_rechazadas_bivariado),
        ("NO_EVALUABLE", variables_no_evaluables_bivariado)
    ] for v in variables
]))
display(df_resumen_bivariado[["Variable", "IV_total", "N_bins_efectivos", "Monotonicidad", "Estado_bivariado", "Motivos"]])
print("Las matrices v4 originales se conservan completas para XGBoost.")

## BIV.13 — Aplicación de WoE a TRAIN, TEST y OOT

**Objetivo:** Aplicar mapas aprendidos en TRAIN sin reajuste y conservar los originales.

**Procedencia:** `3.0.0.bivariado.py`.

**Resultado esperado:** Tres matrices WoE y diagnósticos de categorías/bins no vistos.

Si un bin no tuvo observaciones TRAIN, se usa el prior neutral WoE=0 con conteo explícito; no se estima una tasa nueva con TEST/OOT. Esto es distinto de asignar cero artificial a un bin puro observado. Puede configurar UNSEEN_POLICY="raise".
Las matrices WoE contienen exclusivamente las variables APROBADA de TRAIN, en el orden persistido del JSON. Las originales v4 conservan todas sus columnas.


In [ ]:
# Archivo: 3.0.0.bivariado.py | Función: transform_woe(). No recibe target.
X_train_woe, diagnostico_train_woe = biv.transform_woe(X_train_v4, binning_woe_train)
X_test_woe, diagnostico_test_woe = biv.transform_woe(X_test_v4, binning_woe_train)
X_oot_woe, diagnostico_oot_woe = biv.transform_woe(X_oot_v4, binning_woe_train)
matrices_woe = {"TRAIN": X_train_woe, "TEST": X_test_woe, "OOT": X_oot_woe}
df_diagnosticos_transformacion = pd.concat([
    diagnostico_train_woe.assign(Segmento="TRAIN"),
    diagnostico_test_woe.assign(Segmento="TEST"),
    diagnostico_oot_woe.assign(Segmento="OOT"),
], ignore_index=True)
display(df_diagnosticos_transformacion)
display(X_train_woe.head())

## BIV.14 — Validación de las matrices transformadas

**Objetivo:** Verificar filas, ?ndices, target, segmentos, columnas y valores finitos.

**Procedencia:** `3.0.0.bivariado.py`.

**Resultado esperado:** Contrato validado y originales intactos para futuro XGBoost.



In [ ]:
# Archivo: 3.0.0.bivariado.py | Función: validate_transformed().
df_validacion_woe = biv.validate_transformed(
    muestras_biv["X"], matrices_woe, muestras_biv["y"], muestras_biv["metadata"],
    approved_columns=variables_aprobadas_bivariado
)
display(df_validacion_woe)
# Releer exclusivamente el artefacto persistido para confirmar que X original no cambió.
# Función: load_v4(), mismo archivo 3.0.0.bivariado.py.
originales_verificacion = biv.load_v4(manifest_file=muestras_biv["manifest_file"])
for segmento in ["TRAIN", "TEST", "OOT"]:
    pd.testing.assert_frame_equal(muestras_biv["X"][segmento], originales_verificacion["X"][segmento], check_exact=True)
print("Originales v4 conservados; WoE finito, alineado y sin modificar target/segmentos.")
# No se certifica el tamaño del bin de ausencia como si fuera un intervalo numérico.
aprobadas_biv = df_resumen_bivariado.loc[
    df_resumen_bivariado.Estado_bivariado.eq("APROBADA"), "Variable"
]
ordinarios_aprobados = df_cortes_seleccionados.loc[
    df_cortes_seleccionados.Variable.isin(aprobadas_biv)
    & ~df_cortes_seleccionados.Bin.isin(biv.SPECIAL)
]
assert ordinarios_aprobados.Dist_clientes.ge(PARAMETROS_BIV["MIN_BIN_PCT"] - 1e-12).all()
assert df_resumen_bivariado.loc[
    df_resumen_bivariado.Variable.isin(aprobadas_biv), "N_bins_efectivos"
].ge(2).all()
print("B.3: aprobadas sin bins ordinarios <5%; missing separado y excluido de tendencia.")

assert X_train_woe.columns.tolist() == variables_aprobadas_bivariado
assert X_test_woe.columns.tolist() == variables_aprobadas_bivariado
assert X_oot_woe.columns.tolist() == variables_aprobadas_bivariado
assert not set(variables_rechazadas_bivariado + variables_no_evaluables_bivariado) & set(X_train_woe.columns)
print("WoE de logística: solo aprobadas; originales v4 completos para XGBoost.")


## BIV.15 — Exportación de cortes, WoE, reportes y gráficos

**Objetivo:** Guardar explícitamente resultados ya calculados y comprobar recuperación del JSON.

**Procedencia:** `3.0.0.bivariado.py`.

**Resultado esperado:** Reportes, JSON, matrices WoE y gráficos; ninguna reestimación.



In [ ]:
EXPORTAR_INDIVIDUALES_BIV = True
EXPORTAR_DASHBOARDS_BIV = True
EXPORTAR_ESTABILIDAD_BIV = True
# Archivo: 3.0.0.bivariado.py | Función: export_results().
# Internamente save_binning()/load_binning() y transform_woe() prueban el roundtrip exacto.
archivos_bivariado = biv.export_results(
    df_candidatos_binning, df_bins_detalle, df_resumen_bivariado,
    df_estabilidad_tasas, binning_woe_train, muestras_biv,
    matrices_woe, df_diagnosticos_transformacion,
    individual=EXPORTAR_INDIVIDUALES_BIV,
    dashboards=EXPORTAR_DASHBOARDS_BIV,
    monthly_plots=EXPORTAR_ESTABILIDAD_BIV,
    comparison=df_comparacion_binning,
    sample_stats=df_estadisticas_bivariado_muestras,
    sample_comparison=df_comparacion_bivariado_muestras,
    sample_aggregate=df_resumen_bivariado_muestras,
)
# Archivo: 3.0.0.bivariado.py | Función: write_report(). Deriva cifras de CSV exportados.
ruta_informe_bivariado = biv.write_report()
print(f"{len(archivos_bivariado)} archivos exportados en {biv.ROOT}")
print("Informe:", ruta_informe_bivariado)